# Carga de datos 
Señales y metadatos de sujetos — grupos *Intact* y *Amputated*.

In [1]:
import pandas as pd

Carga de CSVs crudos: señales y metadatos de sujetos, grupos *Intact* y *Amputated*.

In [2]:
# Sujetos sin amputación
intact_signals = pd.read_csv('../data/master/intact_signals.csv')
intact_subjects = pd.read_csv('../data/master/intact_subjects.csv')
# amputados
amputated_signals = pd.read_csv('../data/master/amputated_signals.csv')
amputated_subjects = pd.read_csv('../data/master/amputated_subjects.csv')
# validar dimensiones
print(f'Sujetos sin amputar\nSignals: {intact_signals.shape}')
print(f'Subjects: {intact_subjects.shape}')
print(f'Sujetos sin amputar\nSignals: {amputated_signals.shape}')
print(f'Subjects: {amputated_subjects.shape}')

Sujetos sin amputar
Signals: (273733, 214)
Subjects: (107, 8)
Sujetos sin amputar
Signals: (47668, 214)
Subjects: (13, 16)


Columnas de interés (8 canales × 8 features) y estímulos seleccionados.

In [3]:
meta_cols = ['subject', 'stimulus', 'Database']
features = ['VAR_STD', 'Range', 'Energy', 'Peak_to_Peak', 'VAR', 'Max', 'MNF', 'SSC']
signal_cols = [f'Channel {ch}_{feat}' for ch in range(1, 9) for feat in features]
selected_stimuli = [22, 25, 27, 30, 31, 37, 40]

Filtrado de ventanas por los estímulos seleccionados.

In [4]:
df_raw_intact = intact_signals[intact_signals['stimulus'].isin(selected_stimuli)][meta_cols + signal_cols].copy()
print(f'Ventanas seleccionadas: {len(df_raw_intact)}')
print(f'Columnas: {len(df_raw_intact.columns)} (4 meta + {len(signal_cols)} señal)')

df_raw_amputated = amputated_signals[amputated_signals['stimulus'].isin(selected_stimuli)][meta_cols + signal_cols].copy()
print(f'Ventanas seleccionadas: {len(df_raw_amputated)}')
print(f'Columnas: {len(df_raw_amputated.columns)} (4 meta + {len(signal_cols)} señal)')

Ventanas seleccionadas: 83291
Columnas: 67 (4 meta + 64 señal)
Ventanas seleccionadas: 14314
Columnas: 67 (4 meta + 64 señal)


Cargar y promediar ventanas

In [5]:
def load_and_preprocess(signals, subjects, features, meta_cols, signal_cols, normalize_by='subject'):
    # 1. Filtrar estímulos
    df_raw = signals[meta_cols + signal_cols].copy()
    if selected_stimuli is not None:
        df_raw = df_raw[signals['stimulus'].isin(selected_stimuli)].copy()

    # 2. Merge con metadatos de sujetos
    df_subjects = subjects.rename(columns={'Subject': 'subject'})
    df_raw = df_raw.merge(df_subjects, on=['subject', 'Database'], how='left')

    print(f'DataFrame procesado: {df_raw.shape} ({df_raw["subject"].nunique()} sujetos × {df_raw["stimulus"].nunique()} stimuli)')
    
    return df_raw

In [6]:
def load_and_preprocess_mean(signals, subjects,features,meta_cols, signal_cols):
   
    # Filtrar estímulos
    df_raw = signals[meta_cols + signal_cols].copy()
    if selected_stimuli is not None:
        df_raw = df_raw[signals['stimulus'].isin(selected_stimuli)].copy()

    # Merge con metadatos de sujetos
    df_subjects = subjects.rename(columns={'Subject': 'subject'})
    df_raw = df_raw.merge(df_subjects, on=['subject', 'Database'], how='left')

    # Promediar ventanas por sujeto×estímulo×Database
    df_avg = df_raw.groupby(['subject', 'stimulus', 'Database'])[signal_cols].mean().reset_index()
    df_avg = df_avg.merge(df_subjects, on=['subject', 'Database'], how='left')

    print(f'DataFrame promedio: {df_avg.shape} ({df_avg["subject"].nunique()} sujetos × {df_avg["stimulus"].nunique()} stimuli)')
    return df_avg

In [7]:
def categorize_bmi(subjects):
    subjects['BMI'] = subjects['Weight'] / (subjects['Height'] / 100) ** 2
    bins_imc = [0, 18.5, 25, 30, 100]
    etiquetas_imc = ['Bajo peso', 'Normal', 'Sobrepeso', 'Obesidad']
    subjects['BMI Category'] = pd.cut(subjects['BMI'], bins=bins_imc, labels=etiquetas_imc)
    return subjects

In [8]:
df_intact=categorize_bmi((load_and_preprocess(df_raw_intact, intact_subjects,features, meta_cols, signal_cols)))
df_amputated=categorize_bmi((load_and_preprocess(df_raw_amputated, amputated_subjects, features, meta_cols, signal_cols)))
df_intact_mean=categorize_bmi((load_and_preprocess_mean(df_raw_intact, intact_subjects,features, meta_cols, signal_cols)))
df_amputated_mean=categorize_bmi((load_and_preprocess_mean(df_raw_amputated, amputated_subjects, features, meta_cols, signal_cols)))

DataFrame procesado: (83291, 73) (40 sujetos × 7 stimuli)
DataFrame procesado: (14314, 81) (13 sujetos × 7 stimuli)
DataFrame promedio: (749, 73) (40 sujetos × 7 stimuli)
DataFrame promedio: (86, 81) (13 sujetos × 7 stimuli)


Guardar database

In [9]:
df_intact.to_csv('../data/master/databases_intact_merged.csv', index=False)
df_amputated.to_csv('../data/master/databases_amputated_merged.csv', index=False)
df_intact_mean.to_csv('../data/master/databases_intact_mean.csv', index=False)
df_amputated_mean.to_csv('../data/master/databases_amputated_mean.csv', index=False)

## Tarea 1 — Tablas descriptivas del artículo

In [10]:
# Tabla demográfica: Intact vs Amputated.
def demographics_table(intact_subjects, amputated_subjects):
    """
    Tabla demográfica estilo artículo: Intact vs Amputated.
    Media ± DE [Mín, Máx] para numéricas. Frecuencias para categóricas.
    """
    num_vars = ['Age', 'Height', 'Weight']
    cat_vars = ['Gender', 'Hand', 'Handedness']
    
    rows = []
    
    # Numéricas
    for var in num_vars:
        i = intact_subjects[var].dropna()
        a = amputated_subjects[var].dropna()
        rows.append({
            'Variable': var,
            'Intact (n=107)': f'{i.mean():.1f} ± {i.std():.1f} [{i.min():.0f}, {i.max():.0f}]',
            'Amputated (n=13)': f'{a.mean():.1f} ± {a.std():.1f} [{a.min():.0f}, {a.max():.0f}]'
        })
    
    # BMI
    intact_subjects = intact_subjects.copy()
    amputated_subjects = amputated_subjects.copy()
    intact_subjects['BMI'] = intact_subjects['Weight'] / (intact_subjects['Height']/100)**2
    amputated_subjects['BMI'] = amputated_subjects['Weight'] / (amputated_subjects['Height']/100)**2
    i_bmi = intact_subjects['BMI'].dropna()
    a_bmi = amputated_subjects['BMI'].dropna()
    rows.append({
        'Variable': 'BMI (kg/m²)',
        'Intact (n=107)': f'{i_bmi.mean():.1f} ± {i_bmi.std():.1f} [{i_bmi.min():.1f}, {i_bmi.max():.1f}]',
        'Amputated (n=13)': f'{a_bmi.mean():.1f} ± {a_bmi.std():.1f} [{a_bmi.min():.1f}, {a_bmi.max():.1f}]'
    })
    
    # Categóricas
    for var in cat_vars:
        i_counts = intact_subjects[var].value_counts()
        a_counts = amputated_subjects[var].value_counts()
        i_str = ', '.join([f'{cat}: {n}' for cat, n in i_counts.items()])
        a_str = ', '.join([f'{cat}: {n}' for cat, n in a_counts.items()])
        rows.append({
            'Variable': var,
            'Intact (n=107)': i_str,
            'Amputated (n=13)': a_str
        })
    
    df_table = pd.DataFrame(rows)
    display(df_table)

In [11]:
# Resumen de selección de datos: ventanas, % seleccionado, sujetos.
def data_selection_summary(intact_signals, amputated_signals, selected_stimuli):
    """
    Compara datos originales vs seleccionados:
    ventanas, % seleccionado, sujetos, stimuli.
    """
    rows = []
    
    for name, df_raw in [('Intact', intact_signals), ('Amputated', amputated_signals)]:
        df_sel = df_raw[df_raw['stimulus'].isin(selected_stimuli)]
        
        n_original = len(df_raw)
        n_selected = len(df_sel)
        
        n_subjects = df_raw[['subject', 'Database']].drop_duplicates().shape[0]
        n_subjects_sel = df_sel[['subject', 'Database']].drop_duplicates().shape[0]
        
        n_stimuli_original = df_raw['stimulus'].nunique()
        n_stimuli_selected = df_sel['stimulus'].nunique()
        n_databases = df_raw['Database'].nunique()
        
        rows.append({
            'Grupo': name,
            'Ventanas originales': f'{n_original:,}',
            'Ventanas seleccionadas': f'{n_selected:,}',
            '% Seleccionado': f'{n_selected/n_original*100:.1f}%',
            'Sujetos original': n_subjects,
            'Sujetos seleccionados': n_subjects_sel,
            'Stimuli original': n_stimuli_original,
            'Stimuli seleccionados': n_stimuli_selected,
            'Databases': n_databases
        })
    
    df_table = pd.DataFrame(rows)
    display(df_table)


In [12]:
# **Función maestra:** genera todas las tablas del artículo.
def generate_article_table(intact_subj, amputated_subj, intact_sig, amputated_sig,
                           selected_stimuli):
    """
    Función maestra: genera todas las tablas del artículo.
    """
    tables = {}
    
    print('=' * 60)
    print('Tabla 1: Demografía')
    print('=' * 60)
    tables['demographics'] = demographics_table(intact_subj, amputated_subj)
    
    print('\n' + '=' * 60)
    print('Tabla 2: Selección de datos')
    print('=' * 60)
    tables['selection'] = data_selection_summary(intact_sig, amputated_sig, selected_stimuli)
    
    return tables

In [13]:
tables = generate_article_table(
    intact_subjects, amputated_subjects,
    intact_signals, amputated_signals,
    selected_stimuli=[22, 25, 27, 30, 31, 37, 40]
)

Tabla 1: Demografía


,Variable,Intact (n=107),Amputated (n=13)
0,Age,"28.9 ± 4.8 [20, 54]","42.2 ± 12.2 [28, 67]"
1,Height,"173.9 ± 8.7 [155, 192]","177.5 ± 5.4 [166, 185]"
2,Weight,"70.8 ± 12.3 [44, 105]","79.6 ± 8.3 [66, 95]"
3,BMI (kg/m²),"23.3 ± 3.0 [18.0, 35.2]","25.3 ± 2.5 [21.9, 29.3]"
4,Gender,"Male: 78, Female: 29","Male: 12, Female: 1"
5,Hand,Intact: 107,"Right Hand Amputated: 10, Left Hand Amputated: 3"
6,Handedness,"Right: 94, Left: 13","Right: 12, Left: 1"



Tabla 2: Selección de datos


,Grupo,Ventanas originales,Ventanas seleccionadas,% Seleccionado,Sujetos original,Sujetos seleccionados,Stimuli original,Stimuli seleccionados,Databases
0,Intact,"273,733","83,291",30.4%,107,107,23,7,5
1,Amputated,"47,668","14,314",30.0%,13,13,23,7,2
